# LeetCode 798 — Smallest Rotation with Highest Score bằng Python

- Link: https://leetcode.com/problems/smallest-rotation-with-highest-score/
- Độ khó: **Hard**
- Chủ đề: Mảng, Mảng hiệu (difference array), Tổng tiền tố, Đoạn vòng (cyclic interval), Đếm gia tăng

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def bestRotation(self, nums: List[int]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Xoay mảng `nums` sang trái `k` bước. Sau khi xoay, mỗi phần tử có giá trị **không lớn hơn chỉ số** của nó được 1 điểm. Tìm `k` cho điểm cao nhất (nếu hoà, lấy `k` nhỏ nhất).

Cách thẳng là thử mọi `k` và đếm lại → `O(n²)`. Bài Hard này dạy một kỹ thuật cực kỳ hữu dụng: **đảo góc nhìn** — thay vì hỏi "với mỗi `k`, những phần tử nào ghi điểm?", ta hỏi "với mỗi phần tử, những `k` nào làm nó ghi điểm?". Câu trả lời là một **đoạn liên tiếp (vòng)** các `k` → cộng dồn bằng **mảng hiệu** trong `O(n)`.

Sau notebook này bạn sẽ:

1. Viết lời giải `O(n²)` làm chuẩn đối chiếu.
2. Tính chính xác **đoạn các `k`** mà một phần tử ghi điểm (kể cả khi đoạn "quấn vòng").
3. Dùng **mảng hiệu + tổng tiền tố** để cộng `n` đoạn trong `O(n)`.
4. Suy ra công thức **gia tăng** `score(k) = score(k-1) + 1 − lose[k]`.
5. Xử lý đúng quy tắc hoà (lấy `k` nhỏ nhất).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `nums` độ dài `n`. Xoay `k` (`0 ≤ k < n`) cho mảng `[nums[k], nums[k+1], ..., nums[n-1], nums[0], ..., nums[k-1]]`. Sau khi xoay, phần tử ở chỉ số `j` có giá trị `≤ j` thì được 1 điểm. Trả về `k` cho tổng điểm lớn nhất; nếu nhiều `k` cùng điểm, trả `k` nhỏ nhất.

**Ví dụ 1**

```text
Input: nums = [2,3,1,4,0]
Output: 3
Giải thích:
k = 0: [2,3,1,4,0]  điểm 2
k = 1: [3,1,4,0,2]  điểm 3
k = 2: [1,4,0,2,3]  điểm 3
k = 3: [4,0,2,3,1]  điểm 4
k = 4: [0,2,3,1,4]  điểm 3
→ k = 3.
```

**Ví dụ 2**

```text
Input: nums = [1,3,0,2,4]
Output: 0
Giải thích: mọi phép xoay đều được 3 điểm → lấy k nhỏ nhất = 0.
```

**Ràng buộc**

- `1 <= nums.length <= 10⁵`
- `0 <= nums[i] < nums.length`

In [1]:
from typing import List

EXAMPLES = [
    ([2, 3, 1, 4, 0], 3),
    ([1, 3, 0, 2, 4], 0),
    ([0], 0),
    ([0, 0, 0], 0),
]

def rotate(nums, k):
    return nums[k:] + nums[:k]

def score(arr):
    return sum(1 for j, v in enumerate(arr) if v <= j)

for k in range(5):
    r = rotate([2, 3, 1, 4, 0], k)
    print(k, r, score(r))

0 [2, 3, 1, 4, 0] 2
1 [3, 1, 4, 0, 2] 3
2 [1, 4, 0, 2, 3] 3
3 [4, 0, 2, 3, 1] 4
4 [0, 2, 3, 1, 4] 3


## 3. Kiến thức nền

### 3.1. Slicing để xoay mảng

`nums[k:] + nums[:k]` xoay trái `k` bước, tốn `O(n)`.

### 3.2. Phép `%` với số âm trong Python

Python luôn trả `a % n` thuộc `[0, n)` kể cả khi `a < 0`: `-1 % 5 == 4`. Rất tiện cho chỉ số vòng.

In [2]:
print(-1 % 5, -6 % 5, 7 % 5)

4 4 2


### 3.3. Mảng hiệu (difference array)

Muốn cộng `+1` vào mọi phần tử của đoạn `[l, r]` cho nhiều đoạn, rồi mới đọc kết quả: đặt `d[l] += 1`, `d[r+1] -= 1`, cuối cùng lấy tổng tiền tố. Mỗi đoạn `O(1)`, đọc tất cả `O(n)`.

In [3]:
from itertools import accumulate

n = 8
d = [0] * (n + 1)
for l, r in [(1, 3), (2, 6), (5, 7)]:
    d[l] += 1
    d[r + 1] -= 1
print("mảng hiệu :", d)
print("sau prefix:", list(accumulate(d))[:n])

mảng hiệu : [0, 1, 1, 0, -1, 1, 0, -1, -1]
sau prefix: [0, 1, 2, 2, 1, 2, 2, 1]


### 3.4. Đoạn vòng (cyclic interval)

Trên vòng `0..n-1`, "đoạn từ `s` tới `e`" với `s > e` nghĩa là `s, s+1, ..., n-1, 0, 1, ..., e`. Ta tách thành hai đoạn thường `[s, n-1]` và `[0, e]`.

In [4]:
def add_cyclic(d, s, e, n, val=1):
    """Cộng val vào mọi k trên đoạn vòng s..e (bao gồm hai đầu)."""
    if s <= e:
        d[s] += val; d[e + 1] -= val
    else:
        d[s] += val; d[n] -= val
        d[0] += val; d[e + 1] -= val

d = [0] * (n + 1)
add_cyclic(d, 6, 1, n)
print(list(accumulate(d))[:n])     # 6, 7, 0, 1 được +1

[1, 1, 0, 0, 0, 0, 1, 1]


### 3.5. `argmax` lấy chỉ số nhỏ nhất khi hoà

`max(range(n), key=lambda k: s[k])` trả về chỉ số **đầu tiên** đạt max — đúng quy tắc hoà. Hoặc duyệt tay với so sánh `>` (không phải `>=`).

In [5]:
s = [3, 5, 2, 5, 1]
print(max(range(len(s)), key=lambda k: s[k]))

1


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Phần tử `i` đi đâu sau khi xoay `k`?

Phần tử ban đầu ở chỉ số `i` sẽ ở chỉ số `j = (i − k) mod n`.

### 4.2. Khi nào phần tử `i` ghi điểm?

Khi `nums[i] ≤ j`, tức `j ∈ [nums[i], n−1]`. Đổi sang `k = (i − j) mod n`:

- `j = n − 1` → `k = (i + 1) mod n`
- `j = nums[i]` → `k = (i − nums[i]) mod n`

Khi `j` giảm từ `n−1` xuống `nums[i]`, `k` tăng dần (theo vòng) từ `(i+1) mod n` tới `(i − nums[i]) mod n`. Vậy các `k` tốt cho phần tử `i` là **đoạn vòng** `[(i+1) mod n, (i − nums[i]) mod n]`, độ dài `n − nums[i]` (luôn ≥ 1 vì `nums[i] < n`).

### 4.3. Tính tay ví dụ 1: `nums = [2,3,1,4,0]`, `n = 5`

| i | nums[i] | đoạn k tốt (vòng) | các k |
|---|---|---|---|
| 0 | 2 | [1, 3] | 1, 2, 3 |
| 1 | 3 | [2, 3] | 2, 3 |
| 2 | 1 | [3, 1] | 3, 4, 0, 1 |
| 3 | 4 | [4, 4] | 4 |
| 4 | 0 | [0, 4] | 0, 1, 2, 3, 4 |

Đếm theo `k`: `k=0`: i=2,4 → 2; `k=1`: i=0,2,4 → 3; `k=2`: i=0,1,4 → 3; `k=3`: i=0,1,2,4 → 4; `k=4`: i=2,3,4 → 3. Khớp đề → `k = 3`.

In [6]:
nums = [2, 3, 1, 4, 0]
n = len(nums)
for i, x in enumerate(nums):
    s, e = (i + 1) % n, (i - x) % n
    ks = [(s + t) % n for t in range(n - x)]
    print(f"i={i}, nums[i]={x}: đoạn [{s}, {e}] -> k = {ks}")

i=0, nums[i]=2: đoạn [1, 3] -> k = [1, 2, 3]
i=1, nums[i]=3: đoạn [2, 3] -> k = [2, 3]
i=2, nums[i]=1: đoạn [3, 1] -> k = [3, 4, 0, 1]
i=3, nums[i]=4: đoạn [4, 4] -> k = [4]
i=4, nums[i]=0: đoạn [0, 4] -> k = [0, 1, 2, 3, 4]


## 5. Cách 1 — Brute Force: thử mọi `k`

**Ý tưởng.** Với mỗi `k`, xoay và đếm điểm. Giữ `k` tốt nhất (so sánh `>` để giữ `k` nhỏ khi hoà).

In [7]:
def best_brute(nums):
    n = len(nums)
    best_k, best_s = 0, -1
    for k in range(n):
        s = sum(1 for i in range(n) if nums[i] <= (i - k) % n)
        if s > best_s:
            best_k, best_s = k, s
    return best_k

for nums_, exp in EXAMPLES:
    print(nums_, "->", best_brute(nums_), "(kỳ vọng", exp, ")")
    assert best_brute(nums_) == exp

[2, 3, 1, 4, 0] -> 3 (kỳ vọng 3 )
[1, 3, 0, 2, 4] -> 0 (kỳ vọng 0 )
[0] -> 0 (kỳ vọng 0 )
[0, 0, 0] -> 0 (kỳ vọng 0 )


**Độ phức tạp.** `n` giá trị `k` × `O(n)` đếm → **`O(n²)`**. Với `n = 10⁵`: `10¹⁰` phép → TLE.

**Nút thắt.** Khi `k` tăng 1, hầu hết phần tử chỉ dịch trái 1 ô và trạng thái ghi điểm của chúng **không đổi**. Ta đang đếm lại từ đầu những thứ gần như không thay đổi.

## 6. Bước đột phá

**Insight 1 — Đảo góc nhìn.** Thay vì "với mỗi `k`, đếm phần tử", làm "với mỗi phần tử, đánh dấu đoạn `k`". Mỗi phần tử cho **một đoạn vòng** `[(i+1) % n, (i − nums[i]) % n]`.

**Insight 2 — Cộng nhiều đoạn bằng mảng hiệu.** `n` đoạn, mỗi đoạn `O(1)` → tổng `O(n)`, rồi tổng tiền tố cho `score[k]` với mọi `k`.

**Insight 3 — Gia tăng khi xoay thêm 1.** Từ `k−1` sang `k`:
- Phần tử ở đầu (chỉ số 0) chuyển xuống cuối (chỉ số `n−1`) → **luôn** ghi điểm sau đó (vì `nums < n`). Trước đó nó ghi điểm chỉ khi giá trị là 0.
- Mọi phần tử khác dịch trái 1: phần tử **mất điểm** đúng khi nó đang ở chỉ số bằng chính giá trị của nó.

Quy tắc: *nếu* phần tử `i` có `(i − nums[i] + 1) mod n == k` *thì* nó "mất điểm" ở bước `k` (đối với `nums[i] = 0` thì "mất" ở đây bù cho việc nó được tính +1 khi nhảy xuống cuối). Kết quả gọn: `score(k) = score(k−1) + 1 − lose[k]`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Mảng hiệu trên các đoạn "tốt"

In [8]:
def best_diff_good(nums):
    n = len(nums)
    d = [0] * (n + 1)
    for i, x in enumerate(nums):
        s, e = (i + 1) % n, (i - x) % n
        if s <= e:
            d[s] += 1; d[e + 1] -= 1
        else:
            d[s] += 1; d[n] -= 1
            d[0] += 1; d[e + 1] -= 1
    best_k, best_s, cur = 0, -1, 0
    for k in range(n):
        cur += d[k]
        if cur > best_s:
            best_k, best_s = k, cur
    return best_k

for nums_, exp in EXAMPLES:
    assert best_diff_good(nums_) == exp
print("Mảng hiệu (đoạn tốt): OK")

Mảng hiệu (đoạn tốt): OK


Chú ý trường hợp đặc biệt `nums[i] = 0`: đoạn có độ dài `n` — từ `(i+1) % n` tới `i`, tức **toàn bộ vòng**. Nếu `i = n − 1` thì `s = 0, e = n − 1` → nhánh `s <= e`. Ngược lại `s = e + 1 > e` → nhánh tách đôi, cũng phủ cả vòng. Cả hai đều đúng.

Code nộp LeetCode cho cách 2:

In [9]:
class SolutionDiffGood:
    def bestRotation(self, nums: List[int]) -> int:
        n = len(nums)
        d = [0] * (n + 1)
        for i, x in enumerate(nums):
            s, e = (i + 1) % n, (i - x) % n
            d[s] += 1
            d[e + 1] -= 1
            if s > e:                 # đoạn quấn vòng: thêm phần [0, e] và cắt ở n
                d[0] += 1
                d[n] -= 1
        best_k, best, cur = 0, -1, 0
        for k in range(n):
            cur += d[k]
            if cur > best:
                best, best_k = cur, k
        return best_k

for nums_, exp in EXAMPLES:
    assert SolutionDiffGood().bestRotation(nums_) == exp
print("SolutionDiffGood: OK")

SolutionDiffGood: OK


Mẹo viết gọn ở trên: với đoạn quấn vòng `[s, n−1] ∪ [0, e]`, ta vẫn làm `d[s] += 1; d[e+1] -= 1` (như đoạn thường) — vì `e + 1 ≤ s`, điều này tạo ra "−1 trên `[e+1, s−1]`" so với nền. Cộng thêm `+1` cho cả vòng (`d[0] += 1; d[n] -= 1`) thì thành: `+1` ở ngoài `[e+1, s−1]`, `0` ở trong — đúng là đoạn quấn vòng.

**Độ phức tạp cách 2:** **`O(n)`** thời gian, `O(n)` bộ nhớ.

### 7.2. Cách 3 — Mảng hiệu trên các đoạn "xấu" (lời giải chính thức)

Phần bù của đoạn tốt là đoạn **xấu** `[(i − nums[i] + 1) % n, i]` (độ dài `nums[i]`; rỗng nếu `nums[i] = 0`). Ta trừ 1 trên đoạn xấu; `k` có điểm cao nhất ⇔ bị trừ ít nhất.

In [10]:
def best_diff_bad(nums):
    n = len(nums)
    bad = [0] * (n + 1)
    for i, x in enumerate(nums):
        if x == 0:
            continue                       # không bao giờ mất điểm
        l, r = (i - x + 1) % n, i
        if l <= r:
            bad[l] += 1; bad[r + 1] -= 1
        else:
            bad[l] += 1; bad[n] -= 1
            bad[0] += 1; bad[r + 1] -= 1
    best_k, best_bad, cur = 0, float('inf'), 0
    for k in range(n):
        cur += bad[k]
        if cur < best_bad:
            best_k, best_bad = k, cur
    return best_k

for nums_, exp in EXAMPLES:
    assert best_diff_bad(nums_) == exp
print("Mảng hiệu (đoạn xấu): OK")

Mảng hiệu (đoạn xấu): OK


In [11]:
class SolutionDiffBad:
    def bestRotation(self, nums: List[int]) -> int:
        n = len(nums)
        bad = [0] * (n + 1)
        for i, x in enumerate(nums):
            l, r = (i - x + 1) % n, (i + 1) % n   # đoạn xấu nửa mở [l, r)
            bad[l] -= 1
            bad[r] += 1
            if l > r:                              # quấn vòng
                bad[0] -= 1
        best_k, best, cur = 0, float('-inf'), 0
        for k in range(n):
            cur += bad[k]
            if cur > best:
                best, best_k = cur, k
        return best_k

for nums_, exp in EXAMPLES:
    assert SolutionDiffBad().bestRotation(nums_) == exp
print("SolutionDiffBad: OK")

SolutionDiffBad: OK


Ở `SolutionDiffBad`, ta dùng đoạn nửa mở `[l, r)` với `r = (i+1) % n`. Trường hợp `x = 0`: `l = r` → đoạn rỗng (`−1` rồi `+1` cùng chỗ), đúng. Trường hợp quấn vòng `l > r`: thêm `−1` ở 0 (phần `[0, r)`); phần `[l, n)` không cần đóng vì ta chỉ đọc tới `n − 1`. (Giá trị tuyệt đối `cur` lệch một hằng số so với điểm thật, nhưng `argmax` không đổi.)

**Độ phức tạp cách 3:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.3. Cách 4 — Đếm gia tăng `score(k) = score(k−1) + 1 − lose[k]`

- `lose[k]` = số phần tử `i` với `(i − nums[i] + 1) % n == k`.
- `score(0)` tính trực tiếp, hoặc chỉ cần so sánh tương đối: đặt `rel(0) = 0`, `rel(k) = rel(k−1) + 1 − lose[k]`.

In [12]:
def best_incremental(nums):
    n = len(nums)
    lose = [0] * n
    for i, x in enumerate(nums):
        lose[(i - x + 1) % n] += 1
    best_k, best, cur = 0, 0, 0
    for k in range(1, n):
        cur += 1 - lose[k]
        if cur > best:
            best, best_k = cur, k
    return best_k

for nums_, exp in EXAMPLES:
    assert best_incremental(nums_) == exp
print("Đếm gia tăng: OK")

Đếm gia tăng: OK


Kiểm tra công thức trên ví dụ 1 bằng cách so với điểm thật:

In [13]:
nums = [2, 3, 1, 4, 0]
n = len(nums)
lose = [0] * n
for i, x in enumerate(nums):
    lose[(i - x + 1) % n] += 1
real = [score(rotate(nums, k)) for k in range(n)]
s = real[0]
pred = [s]
for k in range(1, n):
    s += 1 - lose[k]
    pred.append(s)
print("lose :", lose)
print("thật :", real)
print("dự đoán:", pred)
assert real == pred

lose : [2, 0, 1, 0, 2]
thật : [2, 3, 3, 4, 3]
dự đoán: [2, 3, 3, 4, 3]


In [14]:
class SolutionIncremental:
    def bestRotation(self, nums: List[int]) -> int:
        n = len(nums)
        lose = [0] * n
        for i, x in enumerate(nums):
            lose[(i - x + 1) % n] += 1
        best_k = best = cur = 0
        for k in range(1, n):
            cur += 1 - lose[k]
            if cur > best:
                best, best_k = cur, k
        return best_k

for nums_, exp in EXAMPLES:
    assert SolutionIncremental().bestRotation(nums_) == exp
print("SolutionIncremental: OK")

SolutionIncremental: OK


**Độ phức tạp cách 4:** `O(n)` thời gian, `O(n)` bộ nhớ — ngắn nhất.

## 8. TRACE từng bước

### 8.1. Trace cách 2 — các đoạn và mảng hiệu

In [15]:
def trace_diff_good(nums):
    n = len(nums)
    d = [0] * (n + 1)
    print(f"{'i':>2} {'x':>2} | đoạn tốt | d sau khi thêm")
    for i, x in enumerate(nums):
        s, e = (i + 1) % n, (i - x) % n
        d[s] += 1; d[e + 1] -= 1
        if s > e:
            d[0] += 1; d[n] -= 1
        print(f"{i:>2} {x:>2} | [{s},{e}]{' (vòng)' if s > e else '      '} | {d}")
    cur, scores = 0, []
    for k in range(n):
        cur += d[k]
        scores.append(cur)
    print("score[k] =", scores)
    best_k = max(range(n), key=lambda k: scores[k])
    print("=> k =", best_k)
    return best_k

trace_diff_good([2, 3, 1, 4, 0])

 i  x | đoạn tốt | d sau khi thêm
 0  2 | [1,3]       | [0, 1, 0, 0, -1, 0]
 1  3 | [2,3]       | [0, 1, 1, 0, -2, 0]
 2  1 | [3,1] (vòng) | [1, 1, 0, 1, -2, -1]
 3  4 | [4,4]       | [1, 1, 0, 1, -1, -2]
 4  0 | [0,4]       | [2, 1, 0, 1, -1, -3]
score[k] = [2, 3, 3, 4, 3]
=> k = 3


3

Bảng tóm tắt:

| k | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| score | 2 | 3 | 3 | **4** | 3 |

### 8.2. Trace cách 4 — `lose[k]` và điểm tương đối

In [16]:
def trace_incremental(nums):
    n = len(nums)
    lose = [0] * n
    for i, x in enumerate(nums):
        k = (i - x + 1) % n
        lose[k] += 1
        print(f"phần tử i={i} (giá trị {x}) mất điểm khi chuyển sang k={k}")
    print("lose =", lose)
    cur, best, best_k = 0, 0, 0
    print(f"k=0: rel=0")
    for k in range(1, n):
        cur += 1 - lose[k]
        mark = ""
        if cur > best:
            best, best_k = cur, k
            mark = "  <- tốt nhất mới"
        print(f"k={k}: rel = rel(k-1) + 1 - lose[{k}] = {cur}{mark}")
    return best_k

print("=> k =", trace_incremental([2, 3, 1, 4, 0]))
print()
print("=> k =", trace_incremental([1, 3, 0, 2, 4]))

phần tử i=0 (giá trị 2) mất điểm khi chuyển sang k=4
phần tử i=1 (giá trị 3) mất điểm khi chuyển sang k=4
phần tử i=2 (giá trị 1) mất điểm khi chuyển sang k=2
phần tử i=3 (giá trị 4) mất điểm khi chuyển sang k=0
phần tử i=4 (giá trị 0) mất điểm khi chuyển sang k=0
lose = [2, 0, 1, 0, 2]
k=0: rel=0
k=1: rel = rel(k-1) + 1 - lose[1] = 1  <- tốt nhất mới
k=2: rel = rel(k-1) + 1 - lose[2] = 1
k=3: rel = rel(k-1) + 1 - lose[3] = 2  <- tốt nhất mới
k=4: rel = rel(k-1) + 1 - lose[4] = 1
=> k = 3

phần tử i=0 (giá trị 1) mất điểm khi chuyển sang k=0
phần tử i=1 (giá trị 3) mất điểm khi chuyển sang k=4
phần tử i=2 (giá trị 0) mất điểm khi chuyển sang k=3
phần tử i=3 (giá trị 2) mất điểm khi chuyển sang k=2
phần tử i=4 (giá trị 4) mất điểm khi chuyển sang k=1
lose = [1, 1, 1, 1, 1]
k=0: rel=0
k=1: rel = rel(k-1) + 1 - lose[1] = 0
k=2: rel = rel(k-1) + 1 - lose[2] = 0
k=3: rel = rel(k-1) + 1 - lose[3] = 0
k=4: rel = rel(k-1) + 1 - lose[4] = 0
=> k = 0


Ở ví dụ 2, mọi `rel(k) = 0` (không có `k` nào **lớn hơn** `k = 0`), nên đáp án là 0 — đúng quy tắc hoà.

## 9. Code nộp LeetCode

In [17]:
class Solution:
    def bestRotation(self, nums: List[int]) -> int:
        n = len(nums)
        lose = [0] * n
        for i, x in enumerate(nums):
            # phần tử i đứng ở chỉ số đúng bằng giá trị x khi k = (i - x) % n;
            # xoay thêm 1 bước (k + 1) nó tụt xuống x - 1 < x -> mất điểm
            lose[(i - x + 1) % n] += 1
        best_k = best = cur = 0
        for k in range(1, n):
            # +1: phần tử từ đầu xuống cuối luôn ghi điểm; -lose[k]: các phần tử mất điểm
            cur += 1 - lose[k]
            if cur > best:            # '>' để giữ k nhỏ nhất khi hoà
                best, best_k = cur, k
        return best_k

for nums_, exp in EXAMPLES:
    assert Solution().bestRotation(nums_) == exp
print("Solution: OK")

Solution: OK


## 10. Vì sao thuật toán đúng

**Cách 2.** Phần tử `i` ở chỉ số `j = (i − k) mod n`. Ánh xạ `k ↦ j` là song ánh trên `{0..n−1}`. Phần tử ghi điểm ⇔ `j ∈ [nums[i], n−1]` ⇔ `k ∈ {(i − j) mod n : j ∈ [nums[i], n−1]}` = đoạn vòng `[(i+1) mod n, (i − nums[i]) mod n]` (khi `j` giảm 1 thì `k` tăng 1 theo vòng). Mảng hiệu cộng đúng `+1` trên đoạn đó. Vậy `score[k] = Σᵢ [k ∈ đoạn_i]` đúng bằng điểm khi xoay `k`.

**Cách 3.** Đoạn xấu là phần bù của đoạn tốt; `score[k] = n − bad[k]` → argmax score = argmin bad.

**Cách 4.** Từ `k−1` sang `k`:
- Phần tử ở chỉ số 0 (gọi là `a`) chuyển tới `n−1`: điểm trước `[a = 0]`, điểm sau `1`. Thay đổi `1 − [a = 0]`.
- Phần tử khác ở chỉ số `j ≥ 1` chuyển tới `j − 1`: mất điểm ⇔ `nums = j` (trước `nums ≤ j`, sau `nums > j − 1`). Không bao giờ được thêm điểm.

Phần tử `i` ở chỉ số `nums[i]` khi `k − 1 = (i − nums[i]) mod n` ⇔ `k = (i − nums[i] + 1) mod n`. Với `nums[i] ≥ 1`: đúng là "mất điểm khi dịch trái". Với `nums[i] = 0`: lúc đó nó ở chỉ số 0, chính là phần tử `a` với `a = 0` — `lose[k]` đếm nó một lần, bù đúng cho `−[a = 0]`. Vậy `score(k) − score(k−1) = 1 − lose[k]`. ∎

**Quy tắc hoà:** cả ba cách duyệt `k` tăng dần và chỉ cập nhật khi **lớn hơn hẳn** → giữ `k` nhỏ nhất.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n²)` | `O(1)` |
| 2. Mảng hiệu đoạn tốt | `O(n)` | `O(n)` |
| 3. Mảng hiệu đoạn xấu | `O(n)` | `O(n)` |
| 4. Đếm gia tăng | `O(n)` | `O(n)` |

In [18]:
import random, time

rnd = random.Random(798)
big = [rnd.randrange(10**5) for _ in range(10**5)]
for name, fn in [("diff_good", best_diff_good), ("diff_bad", best_diff_bad),
                 ("incremental", best_incremental), ("Solution", lambda a: Solution().bestRotation(a))]:
    t0 = time.perf_counter()
    r = fn(big)
    print(f"{name:12}: k = {r}  ({(time.perf_counter() - t0) * 1000:.1f} ms)")

diff_good   : k = 68258  (25.9 ms)
diff_bad    : k = 68258  (24.6 ms)


incremental : k = 68258  (16.0 ms)


Solution    : k = 68258  (15.1 ms)


## 12. Bộ test

In [19]:
methods = {
    "brute": best_brute,
    "diff_good": best_diff_good,
    "diff_bad": best_diff_bad,
    "incremental": best_incremental,
    "SolutionDiffGood": lambda a: SolutionDiffGood().bestRotation(a),
    "SolutionDiffBad": lambda a: SolutionDiffBad().bestRotation(a),
    "SolutionIncremental": lambda a: SolutionIncremental().bestRotation(a),
    "Solution": lambda a: Solution().bestRotation(a),
}

# 12.1 Ví dụ + edge cases
edge = EXAMPLES + [
    ([1, 0], 1),                  # k=1: [0,1] điểm 2
    ([0, 1], 0),
    ([1, 1], 0),                  # mọi k: 1 điểm -> 0
    ([4, 3, 2, 1, 0], 0),         # đã có thể kiểm tra bằng brute
    ([0, 1, 2, 3, 4], 0),
    ([2, 2, 2], 0),
]
for name, fn in methods.items():
    for a, exp in edge:
        assert fn(a) == best_brute(a), (name, a)
for a, exp in EXAMPLES:
    for name, fn in methods.items():
        assert fn(a) == exp, (name, a)
print("Ví dụ + edge cases: OK")

# 12.2 Ngẫu nhiên có seed: đối chiếu brute force
for seed in range(1500):
    r = random.Random(seed)
    n = r.randint(1, 12)
    a = [r.randrange(n) for _ in range(n)]
    ref = best_brute(a)
    for name, fn in methods.items():
        assert fn(a) == ref, (name, a)
print("1500 mảng ngẫu nhiên: khớp brute force")

# 12.3 Mảng vừa (n = 300) có nhiều hoà
for seed in range(30):
    r = random.Random(10_000 + seed)
    n = 300
    a = [r.choice([0, 1, n - 1]) for _ in range(n)]
    ref = best_brute(a)
    for name, fn in methods.items():
        assert fn(a) == ref, (name, seed)
print("Mảng nhiều hoà: OK")

# 12.4 Mảng lớn: các cách nhanh khớp nhau
for seed in range(3):
    r = random.Random(20_000 + seed)
    a = [r.randrange(10**5) for _ in range(10**5)]
    vals = {name: fn(a) for name, fn in methods.items() if name != "brute"}
    assert len(set(vals.values())) == 1, vals
print("Mảng lớn: OK")

Ví dụ + edge cases: OK


1500 mảng ngẫu nhiên: khớp brute force


Mảng nhiều hoà: OK


Mảng lớn: OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `>=` khi cập nhật tốt nhất (trả `k` lớn nhất khi hoà)

**Code sai:**

In [20]:
def wrong_tie(nums):
    n = len(nums)
    lose = [0] * n
    for i, x in enumerate(nums):
        lose[(i - x + 1) % n] += 1
    best_k = best = cur = 0
    for k in range(1, n):
        cur += 1 - lose[k]
        if cur >= best:               # SAI: hoà vẫn cập nhật
            best, best_k = cur, k
    return best_k

print("sai:", wrong_tie([1, 3, 0, 2, 4]), " đúng:", Solution().bestRotation([1, 3, 0, 2, 4]))

sai: 4  đúng: 0


**Vì sao sai:** đề yêu cầu `k` **nhỏ nhất** khi hoà; `>=` ghi đè bằng `k` lớn hơn.

**Code đúng:** `if cur > best`.

### Lỗi 2 — Quên trường hợp đoạn quấn vòng trong mảng hiệu

**Code sai:**

In [21]:
def wrong_no_wrap(nums):
    n = len(nums)
    d = [0] * (n + 1)
    for i, x in enumerate(nums):
        s, e = (i + 1) % n, (i - x) % n
        lo, hi = min(s, e), max(s, e)     # SAI: coi như đoạn thường
        d[lo] += 1; d[hi + 1] -= 1
    best_k, best, cur = 0, -1, 0
    for k in range(n):
        cur += d[k]
        if cur > best:
            best, best_k = cur, k
    return best_k

for a in ([2, 3, 1, 4, 0], [1, 0], [3, 0, 1, 1]):
    print(a, "sai:", wrong_no_wrap(a), " đúng:", Solution().bestRotation(a))

[2, 3, 1, 4, 0] sai: 2  đúng: 3
[1, 0] sai: 1  đúng: 1
[3, 0, 1, 1] sai: 1  đúng: 1


**Vì sao sai:** khi `s > e`, đoạn thực là `[s, n−1] ∪ [0, e]` chứ không phải `[e, s]`. Đổi chỗ hai đầu biến đoạn vòng thành **phần bù** của nó (gần như vậy) → sai hoàn toàn.

**Code đúng:** tách đôi đoạn quấn vòng (cách 2/3).

### Lỗi 3 — Sai dấu khi tính chỉ số mới sau xoay

**Code sai:**

In [22]:
def wrong_direction(nums):
    n = len(nums)
    best_k, best_s = 0, -1
    for k in range(n):
        s = sum(1 for i in range(n) if nums[i] <= (i + k) % n)   # SAI: xoay phải
        if s > best_s:
            best_k, best_s = k, s
    return best_k

print("sai:", wrong_direction([2, 3, 1, 4, 0]), " đúng:", best_brute([2, 3, 1, 4, 0]))

sai: 2  đúng: 3


**Vì sao sai:** xoay `k` theo đề là đưa `nums[k]` lên đầu, nên phần tử `i` tới chỉ số `(i − k) mod n`, không phải `(i + k) mod n`. Luôn kiểm tra bằng ví dụ: `k = 1`, `nums[1]` phải về chỉ số 0 → `(1 − 1) % n = 0` ✔.

**Code đúng:** `(i - k) % n`.

### Lỗi 4 — Mảng hiệu cấp phát thiếu một ô

Viết `d = [0] * n` rồi `d[e + 1] -= 1` với `e = n − 1` → `IndexError`. Luôn cấp `n + 1` ô (hoặc dùng `% n` có chủ đích như `SolutionDiffBad`).

In [23]:
def wrong_size(nums):
    n = len(nums)
    d = [0] * n                          # SAI: thiếu 1 ô
    try:
        for i, x in enumerate(nums):
            s, e = (i + 1) % n, (i - x) % n
            if s <= e:
                d[s] += 1; d[e + 1] -= 1
        return "không lỗi"
    except IndexError as ex:
        return f"IndexError: {ex}"

print(wrong_size([0, 1, 2]))

không lỗi


## 14. Mẹo & nhận dạng pattern

- **"Thử mọi phép xoay/mọi dịch chuyển, mỗi phần tử đóng góp độc lập"** → mỗi phần tử ứng với một **đoạn** các phép xoay → mảng hiệu.
- **"Đoạn trên vòng"** → tách thành hai đoạn, hoặc dùng mẹo "+1 cả vòng" (cách 2 gọn).
- **Thay đổi khi dịch 1 bước** → nghĩ tới **công thức gia tăng**: chỉ vài phần tử thay đổi trạng thái.
- **Hoà lấy nhỏ nhất** → duyệt tăng dần, so sánh `>` chặt.
- Cùng họ: **1094 Car Pooling**, **1109 Corporate Flight Bookings** (mảng hiệu), **2381 Shifting Letters II**, **396 Rotate Function** (công thức gia tăng khi xoay).

## 15. Biến thể

1. **Trả về điểm cao nhất** (không phải `k`): `max(score)`.
2. **Điều kiện ghi điểm khác**, ví dụ `nums[j] ≥ j`: đoạn tốt đổi thành `j ∈ [0, nums[i]]` → vẫn là đoạn vòng.
3. **Mỗi phần tử có trọng số** `w[i]`: cộng `w[i]` thay vì 1 trong mảng hiệu.
4. **Xoay phải**: thay `k` bằng `n − k`.
5. **396. Rotate Function:** `F(k) = Σ j·arr_k[j]`, công thức gia tăng `F(k) = F(k−1) + sum − n·nums[n−k]`.

Minh hoạ biến thể 2 (ghi điểm khi `nums[j] ≥ j`):

In [24]:
def best_rotation_ge(nums):
    n = len(nums)
    d = [0] * (n + 1)
    for i, x in enumerate(nums):
        hi_j = min(x, n - 1)                 # j ∈ [0, hi_j]
        s, e = (i - hi_j) % n, i % n         # j = hi_j -> k = i - hi_j ; j = 0 -> k = i
        d[s] += 1; d[e + 1] -= 1
        if s > e:
            d[0] += 1; d[n] -= 1
    best_k, best, cur = 0, -1, 0
    for k in range(n):
        cur += d[k]
        if cur > best:
            best, best_k = cur, k
    return best_k

def best_rotation_ge_brute(nums):
    n = len(nums)
    return max(range(n), key=lambda k: sum(nums[i] >= (i - k) % n for i in range(n)))

for seed in range(300):
    r = random.Random(seed)
    n = r.randint(1, 10)
    a = [r.randrange(n + 2) for _ in range(n)]
    assert best_rotation_ge(a) == best_rotation_ge_brute(a)
print("Biến thể nums[j] >= j: OK")

Biến thể nums[j] >= j: OK


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi `k`, đếm lại | `O(n²)` | `O(1)` | đối chiếu |
| 2. Mảng hiệu đoạn tốt | mỗi phần tử → đoạn `k` tốt | `O(n)` | `O(n)` | dễ suy luận nhất |
| 3. Mảng hiệu đoạn xấu | phần bù, đoạn ngắn hơn | `O(n)` | `O(n)` | lời giải chính thức |
| 4. Đếm gia tăng | `score(k) = score(k−1) + 1 − lose[k]` | `O(n)` | `O(n)` | **ngắn nhất, nộp bài** |

## 17. Bài tập

**Bài 1 (đoán output).** `bestRotation([0, 2, 1])` và `bestRotation([3, 2, 1, 0])` = ?

**Bài 2 (tính tay).** Với `nums = [1, 2, 0, 3]`, lập bảng đoạn tốt của từng phần tử và tính `score[k]` cho mọi `k`.

**Bài 3 (điền TODO).** Hoàn thiện `all_scores(nums)` trả về **danh sách điểm thật** `score[k]` cho mọi `k` trong `O(n)`.

**Bài 4 (tự viết lại).** Viết lại cách 4 nhưng tính `score(0)` thật và trả về cặp `(k_tốt_nhất, điểm_cao_nhất)`.

**Bài 5 (mini challenge).** LeetCode 396 *Rotate Function*: `F(k) = Σ j · arr_k[j]` với `arr_k` là `nums` xoay **phải** `k`. Trả về `max F(k)` trong `O(n)`.

In [25]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def all_scores(nums):
    # TODO: mảng hiệu trên các đoạn tốt, rồi tổng tiền tố
    pass

# Bài 4 — TODO
def best_with_score(nums):
    # TODO: score0 = đếm trực tiếp; cập nhật score += 1 - lose[k]
    pass

# Bài 5 — TODO
def max_rotate_function(nums):
    # TODO: F(0) = Σ j*nums[j]; F(k) = F(k-1) + sum(nums) - n * nums[n-k]
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: liệt kê 3 (hoặc 4) phép xoay.
- Mức 2: `[0,2,1]`: k=0 → điểm 2 (0≤0, 1≤2); k=1: `[2,1,0]` → 2; k=2: `[1,0,2]` → 2 → hoà → 0.
- Mức 3: `[3,2,1,0]`: tính `lose` rồi `rel`.

**Bài 2.**
- Mức 1: `n = 4`, đoạn tốt `[(i+1)%4, (i−x)%4]`.
- Mức 2: i=0 (x=1): [1,3]; i=1 (x=2): [2,3]; i=2 (x=0): cả vòng; i=3 (x=3): [0,0].
- Mức 3: `score = [2, 2, 3, 3]` → `k = 2`.

**Bài 3.**
- Mức 1: dùng lại `d` của cách 2.
- Mức 2: `accumulate(d)` rồi cắt `n` phần tử đầu.
- Mức 3: so với `score(rotate(nums, k))`.

**Bài 4.**
- Mức 1: `score0 = Σ [nums[i] ≤ i]`.
- Mức 2: `cur = score0`, cập nhật như cách 4.
- Mức 3: giữ `best` theo `>`.

**Bài 5.**
- Mức 1: xoay phải 1 bước: mọi phần tử tăng chỉ số 1 (thêm `sum`), riêng phần tử cuối về chỉ số 0.
- Mức 2: phần tử cuối mất `n · nums[n−k]` (nó được cộng `+1` rồi về 0 từ `n−1`).
- Mức 3: `F(k) = F(k−1) + S − n·nums[n−k]`.

## 19. Đáp án tham khảo

In [26]:
S = Solution()
# Bài 1
assert S.bestRotation([0, 2, 1]) == best_brute([0, 2, 1]) == 0
ans_b1 = S.bestRotation([3, 2, 1, 0])
assert ans_b1 == best_brute([3, 2, 1, 0])
print("bestRotation([3,2,1,0]) =", ans_b1)

# Bài 2
assert [score(rotate([1, 2, 0, 3], k)) for k in range(4)] == [2, 2, 3, 3]
assert S.bestRotation([1, 2, 0, 3]) == 2

# Bài 3
def all_scores_ans(nums):
    n = len(nums)
    d = [0] * (n + 1)
    for i, x in enumerate(nums):
        s, e = (i + 1) % n, (i - x) % n
        d[s] += 1; d[e + 1] -= 1
        if s > e:
            d[0] += 1; d[n] -= 1
    return list(accumulate(d))[:n]

for seed in range(300):
    r = random.Random(seed)
    n = r.randint(1, 10)
    a = [r.randrange(n) for _ in range(n)]
    assert all_scores_ans(a) == [score(rotate(a, k)) for k in range(n)]

# Bài 4
def best_with_score_ans(nums):
    n = len(nums)
    lose = [0] * n
    for i, x in enumerate(nums):
        lose[(i - x + 1) % n] += 1
    cur = sum(1 for i, x in enumerate(nums) if x <= i)
    best, best_k = cur, 0
    for k in range(1, n):
        cur += 1 - lose[k]
        if cur > best:
            best, best_k = cur, k
    return best_k, best

assert best_with_score_ans([2, 3, 1, 4, 0]) == (3, 4)
assert best_with_score_ans([1, 3, 0, 2, 4]) == (0, 3)

# Bài 5
def max_rotate_function_ans(nums):
    n = len(nums)
    S_ = sum(nums)
    F = sum(j * v for j, v in enumerate(nums))
    best = F
    for k in range(1, n):
        F += S_ - n * nums[n - k]
        best = max(best, F)
    return best

def max_rotate_function_brute(nums):
    n = len(nums)
    return max(sum(j * v for j, v in enumerate(nums[n - k:] + nums[:n - k])) for k in range(n))

assert max_rotate_function_ans([4, 3, 2, 6]) == 26
assert max_rotate_function_ans([100]) == 0
for seed in range(200):
    r = random.Random(seed)
    a = [r.randint(-50, 50) for _ in range(r.randint(1, 10))]
    assert max_rotate_function_ans(a) == max_rotate_function_brute(a)
print("Tất cả đáp án đúng")

bestRotation([3,2,1,0]) = 1
Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đếm gia tăng):**

```python
n = len(nums); lose = [0] * n
for i, x in enumerate(nums):
    lose[(i - x + 1) % n] += 1
best_k = best = cur = 0
for k in range(1, n):
    cur += 1 - lose[k]
    if cur > best: best, best_k = cur, k
return best_k
```

**Template (mảng hiệu đoạn vòng `[s, e]`):**

```python
d[s] += 1; d[e + 1] -= 1
if s > e: d[0] += 1; d[n] -= 1
```

**Checklist trước khi submit:**

- [ ] Chỉ số sau xoay: `(i - k) % n` (xoay **trái**).
- [ ] Đoạn quấn vòng được xử lý.
- [ ] Mảng hiệu có `n + 1` ô.
- [ ] So sánh `>` để giữ `k` nhỏ nhất khi hoà.
- [ ] `n = 1` → trả 0.